# CEE C181 — Intermediate Assignment 1: Simulation framework
**Focused question:** *How does the distance of a minor braking disturbance upstream of a 3→2 lane-drop bottleneck change the AV penetration rate needed to prevent queue spillback?*

Authors: E. Masterjohn & A. Murillo. This notebook implements the complete study from the proposal in one place:

| Proposal item | Where it lives |
|---|---|
| IDM, Eq. (1)–(2), parameter table (human T = 1.2 s, AV T = 0.6 s) | §2 `IDMParams`, `idm_accel` |
| 3 km, 3-lane freeway, lane drop at km 1.5 | §2 `SimConfig`, §3 `FreewaySim` |
| Managed on-ramp with restricted storage queue before the drop | §3 `_ramp`, `SimConfig.ramp_*` |
| 50 % velocity-reduction pulse at 250 m increments upstream | §3 `_maybe_disturb`, `DISTANCES` |
| Stochastic arrivals, AV penetration 0–100 % | §3 `_arrivals`, `PENETRATIONS` |
| Density maps ρ(x,t), queue dissipation time, shockwave velocity | §4 `compute_metrics`, §9 plots |
| "Boundary where spatial proximity overrides penetration benefits" | §8 `required_penetration`, `critical_distance`, regression |

**Workflow:** §1 install → §2–§6 library code (run once) → §7 validate → §8 choose demand → §9 run study → §10–§12 analyse → §13 sensitivity → §14 export.

## 1. Install libraries

In [ ]:
# Run once per environment (JupyterHub: installs into your user site; restart the kernel afterwards if numpy/pandas were upgraded)
%pip install -q numpy pandas scipy matplotlib statsmodels joblib tqdm ipython

`numpy`, `pandas`, `scipy`, `matplotlib` are required. `statsmodels` (logistic regression), `joblib` (parallel runs) and `tqdm` (progress bars) are optional — the notebook falls back to serial/`numpy` versions if they are missing. No other dependency is needed.

## 2. Imports, parameters and configuration

In [ ]:
import os, math, time, warnings, itertools
from dataclasses import dataclass, field, replace, asdict
from typing import Optional, List, Dict, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from scipy import stats

# Optional libraries: the notebook still runs without them (fallbacks are used)
try:
    import statsmodels.api as sm
    import statsmodels.formula.api as smf
    HAVE_SM = True
except Exception:
    HAVE_SM = False
try:
    from joblib import Parallel, delayed
    HAVE_JOBLIB = True
except Exception:
    HAVE_JOBLIB = False
try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(it, **kw):
        return it

try:
    from IPython.display import display
except Exception:
    display = print

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})
print("numpy", np.__version__, "| pandas", pd.__version__, "| statsmodels:", HAVE_SM, "| joblib:", HAVE_JOBLIB)
print("CPU cores available:", os.cpu_count())

In [ ]:
@dataclass
class IDMParams:
    """Intelligent Driver Model parameters (Treiber et al., 2000). Defaults = proposal Section 1."""
    v0: float = 120 / 3.6   # desired velocity [m/s]  (120 km/h = 33.3 m/s)
    T: float = 1.2          # safe time headway [s]   (human 1.2 s, AV/CACC 0.6 s)
    a: float = 1.0          # max acceleration [m/s^2]
    b: float = 1.25         # comfortable deceleration [m/s^2]
    s0: float = 1.0         # minimum jam distance [m]
    s1: float = 10.0        # additional (speed-dependent) jam distance [m]
    delta: float = 4.0      # acceleration exponent [-]


@dataclass
class SimConfig:
    # ---------------- geometry ----------------
    L: float = 3000.0            # segment length [m]
    x_bn: float = 1500.0         # lane-drop location [m]
    n_lanes: int = 3             # lanes upstream of the drop (lane 0 = right lane = the lane that ends)
    veh_len: float = 4.5         # vehicle length [m]
    # ---------------- time ----------------
    dt: float = 0.2              # integration step [s]
    t_end: float = 1200.0        # simulated time [s]
    # ---------------- demand / fleet ----------------
    demand_per_lane: float = 1650.0   # upstream arrivals [veh/h/lane] (Poisson)
    av_penetration: float = 0.0       # fraction of AVs in [0,1]
    human: IDMParams = field(default_factory=lambda: IDMParams(T=1.2))
    av: IDMParams = field(default_factory=lambda: IDMParams(T=0.6))
    v0_cv: float = 0.0                # coefficient of variation of v0 for human drivers (0 = deterministic)
    # ---------------- lane changing (MOBIL-style) ----------------
    lc_interval: float = 0.6          # each vehicle evaluates a lane change every lc_interval [s]
    lc_cooldown: float = 3.0          # time after a lane change before another is allowed [s]
    politeness: float = 0.2
    a_th: float = 0.2                 # incentive threshold [m/s^2]
    b_safe: float = 3.0               # safe braking limit for new follower [m/s^2]
    b_safe_forced: float = 6.0        # relaxed limit for a vehicle forced to merge at the lane end
    lc_min_gap: float = 2.0           # minimum bumper gap to new leader/follower [m]
    avoid_dist: float = 800.0         # drivers do not move INTO the ending lane closer than this to the drop [m]
    merge_len: float = 500.0          # mandatory merge zone length before the drop [m]
    wall_horizon: float = 300.0       # the lane end acts as a stationary obstacle within this distance [m]
    conflict_dist: float = 20.0       # reject simultaneous lane changes into the same gap [m]
    # ---------------- disturbance ----------------
    dist_distance: Optional[float] = None   # metres UPSTREAM of the lane drop; None = baseline (no disturbance)
    t_dist: float = 400.0                   # time the disturbance is triggered [s]
    dist_mode: str = "zone"                 # 'zone'   = localized slow zone: every vehicle inside it is slowed to dist_factor x prevailing speed
                                            # 'vehicle'= a single lead vehicle (first one to reach the point) brakes
    dist_zone_len: float = 100.0            # length of the slow zone [m] (zone mode)
    dist_lanes: Optional[tuple] = None      # lanes affected in zone mode (None = all lanes)
    dist_lane: int = 1                      # lane of the braking vehicle (vehicle mode)
    dist_factor: float = 0.5                # 0.5 = 50 % velocity reduction
    dist_duration: float = 10.0             # pulse duration [s]
    dist_brake_limit: float = 4.0           # max braking used to reach the reduced speed [m/s^2]
    dist_vehicle: str = "human"             # 'human' | 'av' | 'any'  type of disturbed (lead) vehicle
    # ---------------- on-ramp with restricted storage ----------------
    ramp_enabled: bool = True
    ramp_demand: float = 300.0        # ramp arrivals [veh/h]
    ramp_offset: float = 500.0        # ramp merge point is this far upstream of the drop [m]
    ramp_storage: int = 15            # max vehicles that can wait on the ramp (others are blocked)
    ramp_meter_rate: float = 900.0    # max release rate [veh/h]
    ramp_v: float = 20.0              # max ramp entry speed [m/s]
    ramp_b_safe: float = 4.0
    # ---------------- measurement ----------------
    dx: float = 50.0                  # space bin [m]
    dt_bin: float = 10.0              # time bin [s]
    v_queue: float = 40 / 3.6         # speed below which a cell is 'congested' [m/s]
    queue_bridge: int = 1             # tolerated non-congested bins inside a queue
    spill_len: float = 1000.0         # queue length that counts as 'spillback' [m]
    q_clear: float = 100.0            # queue length regarded as 'dissipated' [m]
    det_offset: float = 100.0         # discharge detector this far downstream of the drop [m]
    seed: int = 0


def lane_capacity_idm(p: IDMParams, veh_len: float = 4.5, n=400):
    """Equilibrium (steady-state) IDM fundamental diagram. Returns density [veh/km], flow [veh/h], speed [m/s]."""
    v = np.linspace(0.1, p.v0 * 0.9999, n)
    s_e = (p.s0 + p.s1 * np.sqrt(v / p.v0) + v * p.T) / np.sqrt(1 - (v / p.v0) ** p.delta)
    rho = 1000.0 / (s_e + veh_len)
    return rho, rho * v * 3.6, v

**Modelling assumptions I had to add** (the proposal does not specify them — all are fields of `SimConfig` so you can change and defend them):
- **Lane changing:** MOBIL-style (politeness 0.2, threshold 0.2 m/s²). Drivers avoid *entering* the ending lane within 800 m of the drop; vehicles already in it must merge in the last 500 m with progressively relaxed safety limits. The lane end acts as a stationary obstacle within 300 m.
- **AVs differ from humans only through the IDM parameters** (here T = 0.6 s) — as in the proposal. Real "coordinated velocity recovery" (CACC / traffic-smoothing controllers) is *not* modelled; this is a limitation worth stating in the report.
- **Disturbance:** default `dist_mode="zone"` = every vehicle in a 100 m zone is slowed to 50 % of the prevailing speed for 10 s (a single braking vehicle is mostly absorbed by lane changing on a 3-lane road). `dist_mode="vehicle"` reproduces the single-lead-vehicle variant.
- **Queue / spillback:** a queue is the contiguous run of 50 m × 10 s cells with speed < 40 km/h starting just upstream of the lane drop; **spillback = queue ≥ 1000 m**. Both thresholds can be re-evaluated afterwards without re-simulating (`reclassify`, §13).

## 3. Microscopic simulator

In [ ]:
def idm_accel(v, s, dv, v0, T, a, b, s0, s1, delta):
    """IDM acceleration, Eq. (1)-(2) of the proposal (vectorised).
    v: speed, s: bumper gap, dv: approach rate (v - v_lead)."""
    s = np.maximum(s, 0.1)
    v = np.maximum(v, 0.0)
    s_star = s0 + s1 * np.sqrt(v / v0) + np.maximum(0.0, v * T + v * dv / (2.0 * np.sqrt(a * b)))
    return a * (1.0 - (v / v0) ** delta - (s_star / s) ** 2)


class FreewaySim:
    """Microscopic multi-lane freeway with a single-lane-drop bottleneck.

    Lane 0 (right lane) ends at x = x_bn. Vehicles follow the IDM; lane changes follow a MOBIL-style rule with a
    mandatory merge for the ending lane. Optional on-ramp with finite storage, and a localised braking disturbance.
    x is the FRONT bumper position [m]; gap = x_lead - veh_len - x.
    """
    CAP = 6000
    PCOLS = ("v0", "T", "a", "b", "s0", "s1", "delta")

    def __init__(self, cfg: SimConfig):
        self.cfg = c = cfg
        self.n_steps = int(round(c.t_end / c.dt))
        self.nx = int(round(c.L / c.dx))
        self.nt = int(math.ceil(c.t_end / c.dt_bin))
        self.lc_every = max(1, int(round(c.lc_interval / c.dt)))

        # common random numbers: arrival streams depend only on the seed, never on the simulation state
        ss = np.random.SeedSequence(c.seed).spawn(2)
        r_main, r_ramp = np.random.default_rng(ss[0]), np.random.default_rng(ss[1])
        lam = c.demand_per_lane / 3600.0 * c.dt
        self.arr_main = r_main.poisson(lam, size=(self.n_steps, c.n_lanes))
        tot = int(self.arr_main.sum()) + 1
        self.u_main, self.z_main = r_main.random(tot), r_main.standard_normal(tot)
        self.arr_ramp = r_ramp.poisson(c.ramp_demand / 3600.0 * c.dt, size=self.n_steps)
        tot_r = int(self.arr_ramp.sum()) + 1
        self.u_ramp = r_ramp.random(tot_r)
        self.ptr_main = 0
        self.ptr_ramp = 0

        # vehicle state arrays
        C = self.CAP
        self.n = 0
        self.x = np.zeros(C); self.v = np.zeros(C)
        self.lane = np.zeros(C, dtype=int); self.av = np.zeros(C, dtype=bool)
        self.t_in = np.zeros(C); self.vid = np.zeros(C, dtype=int)
        self.cool = np.zeros(C); self.pulse_end = np.full(C, -1.0); self.pulse_vt = np.zeros(C)
        self.par = {k: np.zeros(C) for k in self.PCOLS}
        self.next_vid = 0

        # queues
        self.pending = [[] for _ in range(c.n_lanes)]   # virtual entry queues (t_arrival, is_av, z)
        self.ramp_q = []
        self.ramp_last = -1e9
        self.ramp_blocked = 0
        self.ramp_admitted = 0

        # accumulators
        self.sumv = np.zeros((self.nt, self.nx)); self.cnt = np.zeros((self.nt, self.nx))
        self.steps_in_bin = np.zeros(self.nt)
        self.det_bn = np.zeros(self.nt); self.det_exit = np.zeros(self.nt)
        self.pend_ts = np.zeros(self.nt); self.ramp_ts = np.zeros(self.nt)
        self.tt_exited = []; self.exit_av = []
        self.overlap_log = []
        self.min_gap = np.inf; self.overlaps = 0; self.n_lc = 0; self.n_forced_lc = 0
        self.t = 0.0
        self.dist_done = False; self.dist_info = {}

    # ------------------------------------------------------------------ vehicles
    def _add(self, x, v, lane, is_av, t_in, z=0.0):
        i = self.n
        if i >= self.CAP:
            raise RuntimeError("vehicle capacity exceeded; increase FreewaySim.CAP")
        p = self.cfg.av if is_av else self.cfg.human
        # tiny deterministic offset so that vehicles inserted in the same step never share an identical position
        self.x[i], self.v[i], self.lane[i], self.av[i], self.t_in[i] = x + 1e-4 * (self.next_vid % 997), v, lane, is_av, t_in
        self.vid[i] = self.next_vid; self.next_vid += 1
        self.cool[i] = self.cfg.lc_cooldown; self.pulse_end[i] = -1.0   # no lane change right after insertion
        for k in self.PCOLS:
            self.par[k][i] = getattr(p, k)
        if (not is_av) and self.cfg.v0_cv > 0:
            self.par["v0"][i] = max(15.0, p.v0 * (1 + self.cfg.v0_cv * z))
        self.n += 1
        return i

    def _remove(self, keep):
        m = int(keep.sum())
        for arr in (self.x, self.v, self.lane, self.av, self.t_in, self.vid, self.cool, self.pulse_end, self.pulse_vt):
            arr[:m] = arr[:self.n][keep]
        for k in self.PCOLS:
            self.par[k][:m] = self.par[k][:self.n][keep]
        self.n = m

    # ------------------------------------------------------------------ neighbours
    def _lane_struct(self):
        n = self.n; lane = self.lane[:n]; x = self.x[:n]; out = []
        for L in range(self.cfg.n_lanes):
            mem = np.flatnonzero(lane == L)
            order = mem[np.argsort(x[mem], kind="stable")]
            out.append((order, x[order]))
        return out

    @staticmethod
    def _neighbors(xq, lane_q, ls):
        """Leader / follower index (in lane lane_q) for query positions xq. -1 if none."""
        m = len(xq)
        lead = np.full(m, -1); fol = np.full(m, -1)
        for L, (order, xs) in enumerate(ls):
            if order.size == 0:
                continue
            q = np.flatnonzero(lane_q == L)
            if q.size == 0:
                continue
            jr = np.searchsorted(xs, xq[q], side="right")
            jl = np.searchsorted(xs, xq[q], side="left") - 1
            lead[q] = np.where(jr < order.size, order[np.minimum(jr, order.size - 1)], -1)
            fol[q] = np.where(jl >= 0, order[np.maximum(jl, 0)], -1)
        return lead, fol

    # ------------------------------------------------------------------ IDM wrappers
    def _acc_core(self, f, xl, vl, has, wall):
        c = self.cfg
        x, v = self.x[f], self.v[f]
        gap = np.where(has, xl - c.veh_len - x, 1e9)
        dv = np.where(has, v - vl, 0.0)
        if wall is not None:
            wgap = c.x_bn - x
            use = wall & (wgap < c.wall_horizon) & (wgap < gap)
            gap = np.where(use, wgap, gap)
            dv = np.where(use, v, dv)
        P = self.par
        return idm_accel(v, gap, dv, P["v0"][f], P["T"][f], P["a"][f], P["b"][f], P["s0"][f], P["s1"][f], P["delta"][f])

    def _acc_idx(self, f, l, wall):
        has = l >= 0
        li = np.maximum(l, 0)
        return self._acc_core(f, self.x[li], self.v[li], has, wall)

    # ------------------------------------------------------------------ demand
    def _arrivals(self, k):
        c = self.cfg
        for L in range(c.n_lanes):
            for _ in range(int(self.arr_main[k, L])):
                u, z = self.u_main[self.ptr_main], self.z_main[self.ptr_main]
                self.ptr_main += 1
                self.pending[L].append((self.t, bool(u < c.av_penetration), z))
        for _ in range(int(self.arr_ramp[k])):
            u = self.u_ramp[self.ptr_ramp]; self.ptr_ramp += 1
            if not c.ramp_enabled:
                continue
            if len(self.ramp_q) < c.ramp_storage:
                self.ramp_q.append((self.t, bool(u < c.av_penetration)))
                self.ramp_admitted += 1
            else:
                self.ramp_blocked += 1

    @staticmethod
    def _eq_gap(p, v):
        """Desired (equilibrium) bumper gap at speed v with zero approach rate."""
        return p.s0 + p.s1 * math.sqrt(max(v, 0.0) / p.v0) + v * p.T

    def _entry_speed(self, p, gap, v_l, has_leader, vcap=None):
        """Speed at which a vehicle can be inserted behind a leader without disturbing it:
        match the leader's speed (or ~95% of v0 on an empty road) and insert only if the gap is at least the
        IDM equilibrium gap for that speed. Returns None when there is no room yet."""
        v_in = min(v_l, p.v0) if has_leader else 0.95 * p.v0
        if vcap is not None:
            v_in = min(v_in, vcap)
        v_in = min(v_in, math.sqrt(max(v_l, 0.0) ** 2 + 2 * p.b * max(gap - p.s0, 0.0)))
        if gap < max(self._eq_gap(p, v_in), p.s0 + 1.0):
            return None
        return max(v_in, 0.0)

    def _insert_mainline(self):
        c = self.cfg
        for L in range(c.n_lanes):
            if not self.pending[L]:
                continue
            idx = np.flatnonzero(self.lane[:self.n] == L)
            if idx.size:
                j = idx[np.argmin(self.x[idx])]
                gap = self.x[j] - c.veh_len - 0.0
                v_l, has = self.v[j], True
            else:
                gap, v_l, has = 1e9, c.human.v0, False
            t_arr, is_av, z = self.pending[L][0]
            p = c.av if is_av else c.human
            v_in = self._entry_speed(p, gap, v_l, has)
            if v_in is None:
                continue
            self._add(0.0, v_in, L, is_av, t_arr, z)
            self.pending[L].pop(0)

    def _ramp(self):
        c = self.cfg
        if not (c.ramp_enabled and self.ramp_q):
            return
        if self.t - self.ramp_last < 3600.0 / c.ramp_meter_rate:
            return
        t_arr, is_av = self.ramp_q[0]
        p = c.av if is_av else c.human
        xr = c.x_bn - c.ramp_offset
        idx = np.flatnonzero(self.lane[:self.n] == 0)
        gap_l, v_l, gap_f, v_f, f, has_l = 1e9, p.v0, 1e9, 0.0, -1, False
        if idx.size:
            xs = self.x[idx]
            ahead = idx[xs >= xr]; behind = idx[xs < xr]
            if ahead.size:
                j = ahead[np.argmin(self.x[ahead])]; gap_l = self.x[j] - c.veh_len - xr; v_l = self.v[j]; has_l = True
            if behind.size:
                f = behind[np.argmax(self.x[behind])]; gap_f = xr - c.veh_len - self.x[f]; v_f = self.v[f]
        if gap_f < 2.0:
            return
        v_in = self._entry_speed(p, gap_l, v_l, has_l, c.ramp_v)
        if v_in is None:
            return
        if f >= 0:
            a_f = self._acc_core(np.array([f]), np.array([xr]), np.array([v_in]), np.array([True]), np.array([True]))[0]
            if a_f < -c.ramp_b_safe:
                return
        self._add(xr, v_in, 0, is_av, t_arr)
        self.ramp_q.pop(0); self.ramp_last = self.t

    # ------------------------------------------------------------------ disturbance
    def _maybe_disturb(self):
        c = self.cfg
        if c.dist_distance is None or self.dist_done or self.t < c.t_dist:
            return
        xd = c.x_bn - c.dist_distance
        n = self.n
        if c.dist_mode == "zone":
            near = (self.x[:n] >= xd - 200.0) & (self.x[:n] < xd + c.dist_zone_len)
            v_ref = float(self.v[:n][near].mean()) if near.any() else c.human.v0
            self.dist_done = True
            self.zone = dict(x0=xd, x1=xd + c.dist_zone_len, t0=self.t, t1=self.t + c.dist_duration, vt=c.dist_factor * v_ref)
            self.dist_info = dict(t0=self.t, x0=xd, v0=v_ref, is_av=False, vid=-1)
            return
        base = (self.lane[:n] == c.dist_lane) & (self.x[:n] >= xd) & (self.x[:n] < xd + 150.0)
        if c.dist_vehicle == "human":
            ok = base & ~self.av[:n]
        elif c.dist_vehicle == "av":
            ok = base & self.av[:n]
        else:
            ok = base
        if not ok.any() and (c.av_penetration >= 1.0 or c.dist_vehicle == "any" or self.t > c.t_dist + 60):
            ok = base
        cand = np.flatnonzero(ok)
        if cand.size == 0:
            return
        i = cand[np.argmin(self.x[cand] - xd)]
        self.pulse_end[i] = self.t + c.dist_duration
        self.pulse_vt[i] = c.dist_factor * self.v[i]
        self.cool[i] = c.dist_duration + c.lc_cooldown
        self.dist_done = True
        self.dist_info = dict(t0=self.t, x0=self.x[i], v0=self.v[i], is_av=bool(self.av[i]), vid=int(self.vid[i]))

    # ------------------------------------------------------------------ lane changes
    def _lane_changes(self, k):
        c = self.cfg
        n = self.n
        if n < 3:
            return
        x, v, lane = self.x[:n], self.v[:n], self.lane[:n]
        due = (self.vid[:n] % self.lc_every == k % self.lc_every) & (self.cool[:n] <= 0)
        cand = np.flatnonzero(due)
        if cand.size == 0:
            return
        ls = self._lane_struct()
        lead_cur, fol_cur = self._neighbors(x, lane, ls)
        m = cand.size
        xc, lc_ = x[cand], lane[cand]
        in0 = lc_ == 0
        u = np.clip((xc - (c.x_bn - c.merge_len)) / c.merge_len, 0.0, 1.0)
        forced = in0 & (xc > c.x_bn - c.merge_len)
        b_eff = c.b_safe + u * (c.b_safe_forced - c.b_safe) * forced
        p_eff = np.where(forced, 0.0, c.politeness)
        a_old = self._acc_idx(cand, lead_cur[cand], in0)

        score = np.full((2, m), -np.inf)
        for di, d in enumerate((-1, +1)):
            tl = lc_ + d
            valid = (tl >= 0) & (tl < c.n_lanes)
            valid &= ~((tl == 0) & (xc > c.x_bn - c.avoid_dist))     # do not enter the ending lane near the drop
            if d == -1:
                valid &= ~forced                                       # forced vehicles only move left
            vi = np.flatnonzero(valid)
            if vi.size == 0:
                continue
            cc = cand[vi]; tlv = tl[vi]; xcc = x[cc]; vcc = v[cc]
            ln, fn = self._neighbors(xcc, tlv, ls)
            has_l, has_f = ln >= 0, fn >= 0
            lidx, fidx = np.maximum(ln, 0), np.maximum(fn, 0)
            gap_l = np.where(has_l, x[lidx] - c.veh_len - xcc, 1e9)
            gap_f = np.where(has_f, xcc - c.veh_len - x[fidx], 1e9)
            wall_t = tlv == 0
            a_new = self._acc_idx(cc, ln, wall_t)
            a_nf_after = self._acc_core(fidx, xcc, vcc, np.ones(len(cc), bool), wall_t)
            a_nf_before = self._acc_idx(fidx, ln, wall_t)
            of = fol_cur[cc]; has_of = of >= 0; ofi = np.maximum(of, 0)
            w_old = lane[cc] == 0
            a_of_before = self._acc_core(ofi, xcc, vcc, np.ones(len(cc), bool), w_old)
            a_of_after = self._acc_idx(ofi, lead_cur[cc], w_old)
            safe = (gap_l >= c.lc_min_gap) & (gap_f >= c.lc_min_gap) & (a_new >= -b_eff[vi])
            safe &= np.where(has_f, a_nf_after >= -b_eff[vi], True)
            inc = (a_new - a_old[vi]) + p_eff[vi] * (np.where(has_f, a_nf_after - a_nf_before, 0.0)
                                                    + np.where(has_of, a_of_after - a_of_before, 0.0))
            ok = safe & (forced[vi] | (inc > c.a_th))
            sc = np.where(forced[vi], 1e3 + u[vi], inc)
            score[di, vi[ok]] = sc[ok]

        best = np.argmax(score, axis=0)
        bs = score[best, np.arange(m)]
        mv = np.flatnonzero(np.isfinite(bs))
        if mv.size == 0:
            return
        tgt = lc_[mv] + np.where(best[mv] == 1, 1, -1)
        order = np.lexsort((xc[mv], tgt))
        last = {}
        for o in order:
            j = mv[o]; tl = int(tgt[o]); xx = xc[j]
            if tl in last and abs(xx - last[tl]) < c.conflict_dist:
                continue
            last[tl] = xx
            gi = cand[j]
            self.lane[gi] = tl
            self.cool[gi] = c.lc_cooldown
            self.n_lc += 1
            self.n_forced_lc += int(forced[j])

    # ------------------------------------------------------------------ dynamics
    def _dynamics(self, tb):
        c = self.cfg; n = self.n; dt = c.dt
        if n == 0:
            return
        x, v, lane = self.x[:n], self.v[:n], self.lane[:n]
        ls = self._lane_struct()
        lead, _ = self._neighbors(x, lane, ls)
        acc = self._acc_idx(np.arange(n), lead, lane == 0)
        acc = np.maximum(acc, -9.0)
        # braking pulse on the disturbed vehicle
        pm = self.pulse_end[:n] > self.t
        if pm.any():
            ap = np.clip((self.pulse_vt[:n][pm] - v[pm]) / 1.0, -c.dist_brake_limit, 1.0)
            acc[pm] = np.minimum(acc[pm], ap)
        z = getattr(self, "zone", None)
        if z is not None and z["t0"] <= self.t < z["t1"]:
            zm = (x >= z["x0"]) & (x < z["x1"])
            if c.dist_lanes is not None:
                zm &= np.isin(lane, c.dist_lanes)
            if zm.any():
                az = np.clip((z["vt"] - v[zm]) / 1.0, -c.dist_brake_limit, 1.0)
                acc[zm] = np.minimum(acc[zm], az)
        v_new = v + acc * dt
        stop = v_new < 0
        dx = np.where(stop, -v * v / (2 * np.minimum(acc, -1e-6)), v * dt + 0.5 * acc * dt * dt)
        x_new = x + dx
        v_new = np.maximum(v_new, 0.0)
        # the lane end is a hard wall
        w = lane == 0
        x_new = np.where(w, np.minimum(x_new, c.x_bn - 0.5), x_new)
        v_new = np.where(w & (x_new >= c.x_bn - 0.5), 0.0, v_new)
        # detector crossings (count before overwriting x)
        xd = c.x_bn + c.det_offset
        self.det_bn[tb] += int(((x < xd) & (x_new >= xd)).sum())
        self.x[:n], self.v[:n] = x_new, v_new
        self.cool[:n] -= dt
        # overlap / min-gap bookkeeping (and repair)
        for L in range(c.n_lanes):
            mem = np.flatnonzero(self.lane[:n] == L)
            if mem.size < 2:
                continue
            o = mem[np.argsort(self.x[mem], kind="stable")]
            g = self.x[o[1:]] - c.veh_len - self.x[o[:-1]]
            self.min_gap = min(self.min_gap, float(g.min()))
            bad = np.flatnonzero(g < 0.0)
            if bad.size:
                self.overlaps += int(bad.size)
                self.overlap_log.append((round(self.t, 1), L, float(self.x[o[bad[0]]]), float(g[bad[0]])))
                for bi in bad[::-1]:
                    f, l = o[bi], o[bi + 1]
                    self.x[f] = self.x[l] - c.veh_len - 0.05
                    self.v[f] = min(self.v[f], self.v[l])

    def _record(self, tb):
        c = self.cfg; n = self.n
        if n:
            xb = np.clip((self.x[:n] // c.dx).astype(int), 0, self.nx - 1)
            self.sumv[tb] += np.bincount(xb, weights=self.v[:n], minlength=self.nx)
            self.cnt[tb] += np.bincount(xb, minlength=self.nx)
        self.steps_in_bin[tb] += 1
        self.pend_ts[tb] += sum(len(p) for p in self.pending)
        self.ramp_ts[tb] += len(self.ramp_q)

    def _exit(self, tb):
        n = self.n
        gone = self.x[:n] >= self.cfg.L
        if gone.any():
            self.tt_exited.extend((self.t + self.cfg.dt - self.t_in[:n][gone]).tolist())
            self.exit_av.extend(self.av[:n][gone].tolist())
            self.det_exit[tb] += int(gone.sum())
            self._remove(~gone)

    # ------------------------------------------------------------------ main loop
    def step(self, k):
        c = self.cfg
        tb = min(int(self.t // c.dt_bin), self.nt - 1)
        self._arrivals(k)
        self._insert_mainline()
        self._ramp()
        self._maybe_disturb()
        if self.n:
            self._lane_changes(k)
        self._dynamics(tb)
        self._record(tb)
        self._exit(tb)
        self.t += c.dt

    def run(self):
        for k in range(self.n_steps):
            self.step(k)
        return self.results()

    def results(self):
        c = self.cfg
        with np.errstate(invalid="ignore", divide="ignore"):
            V = np.where(self.cnt > 0, self.sumv / self.cnt, np.nan)
            lanes_x = np.where((np.arange(self.nx) + 0.5) * c.dx < c.x_bn, c.n_lanes, c.n_lanes - 1)
            rho = self.cnt / self.steps_in_bin[:, None] / c.dx * 1000.0 / lanes_x[None, :]   # veh/km/lane
            Q = rho * np.nan_to_num(V) * 3.6                                                  # veh/h/lane
        n = self.n
        still = (c.t_end - self.t_in[:n]).sum() if n else 0.0
        waiting = sum(c.t_end - p[0] for q in self.pending for p in q) + sum(c.t_end - p[0] for p in self.ramp_q)
        ttt = (np.sum(self.tt_exited) + still + waiting) / 3600.0
        return dict(V=V, rho=rho, Q=Q, det_bn=self.det_bn.copy(), det_exit=self.det_exit.copy(),
                    pend_ts=self.pend_ts / np.maximum(self.steps_in_bin, 1), ramp_ts=self.ramp_ts / np.maximum(self.steps_in_bin, 1),
                    n_exited=len(self.tt_exited), mean_tt=float(np.mean(self.tt_exited)) if self.tt_exited else np.nan,
                    ttt_h=float(ttt), min_gap=float(self.min_gap), overlaps=int(self.overlaps),
                    n_lc=int(self.n_lc), n_forced_lc=int(self.n_forced_lc),
                    ramp_blocked=int(self.ramp_blocked), ramp_admitted=int(self.ramp_admitted),
                    dist_info=dict(self.dist_info), n_on_road=int(self.n))

## 4. Outcome metrics (queue length, dissipation time, shockwave speed, …)

In [ ]:
def queue_profile(V, cfg: SimConfig):
    """Queue tail position and length over time from the speed map V[t, x].
    A queue = contiguous run of congested cells (speed < v_queue) that starts in the cell just upstream of the lane drop."""
    nt, nx = V.shape
    Vf = np.where(np.isnan(V), cfg.human.v0, V)
    cong = Vf < cfg.v_queue
    ib = int(round(cfg.x_bn / cfg.dx)) - 1
    tail = np.full(nt, cfg.x_bn)
    for ti in range(nt):
        if not cong[ti, ib]:
            continue
        j, miss, last = ib, 0, ib
        while j >= 0:
            if cong[ti, j]:
                last, miss = j, 0
            else:
                miss += 1
                if miss > cfg.queue_bridge:
                    break
            j -= 1
        tail[ti] = last * cfg.dx
    qlen = cfg.x_bn - tail
    return tail, qlen


def compute_metrics(res, cfg: SimConfig):
    """Scalar outcome metrics for one run (all measured from the disturbance time onward)."""
    nt = res["V"].shape[0]
    tc = (np.arange(nt) + 0.5) * cfg.dt_bin
    tail, qlen = queue_profile(res["V"], cfg)
    post = tc >= cfg.t_dist
    qp, tp, taip = qlen[post], tc[post], tail[post]
    i_pk = int(np.argmax(qp)); q_max = float(qp[i_pk]); t_pk = float(tp[i_pk])
    pre = (tc >= cfg.t_dist - 60) & (tc < cfg.t_dist)
    q_pre = float(qlen[pre].mean()) if pre.any() else np.nan

    spill = q_max >= cfg.spill_len
    hit = np.flatnonzero(qp >= cfg.spill_len)
    t_spill = float(tp[hit[0]] - cfg.t_dist) if hit.size else np.nan

    # dissipation: first time after the peak that the queue is shorter than q_clear
    after = np.flatnonzero((tp > t_pk) & (qp <= cfg.q_clear))
    if q_max <= cfg.q_clear:
        t_diss, censored = 0.0, False
    elif after.size:
        t_diss, censored = float(tp[after[0]] - cfg.t_dist), False
    else:
        t_diss, censored = float(cfg.t_end - cfg.t_dist), True

    # shockwave speed of the queue tail while the queue grows (negative = moves upstream)
    grow = (tp <= t_pk) & (qp > 0)
    if grow.sum() >= 3 and np.ptp(tp[grow]) > 0:
        slope = float(np.polyfit(tp[grow], taip[grow], 1)[0])
    else:
        slope = np.nan

    win = (tc >= cfg.t_dist)
    dur = max(win.sum() * cfg.dt_bin, 1.0)
    thru = float(res["det_bn"][win].sum() / dur * 3600.0)       # veh/h across the 2 remaining lanes
    out = dict(q_max=q_max, q_pre=q_pre, spill=bool(spill), t_spill=t_spill, t_peak=t_pk - cfg.t_dist,
               t_diss=t_diss, diss_censored=bool(censored), shock_speed=slope,
               shock_kmh=slope * 3.6 if np.isfinite(slope) else np.nan,
               q_end=float(qlen[-3:].mean()), thru_vph=thru, thru_vph_lane=thru / (cfg.n_lanes - 1),
               mean_tt=res["mean_tt"], ttt_h=res["ttt_h"], n_exited=res["n_exited"],
               pend_max=float(res["pend_ts"].max()), ramp_q_mean=float(res["ramp_ts"].mean()),
               ramp_blocked=res["ramp_blocked"], min_gap=res["min_gap"], overlaps=res["overlaps"],
               n_lc=res["n_lc"], dist_applied=bool(res["dist_info"]), dist_t=res["dist_info"].get("t0", np.nan))
    return out, tail, qlen

## 5. Experiment runner (common random numbers, parallel)

In [ ]:
def _run_one(job):
    cfg, tag = job
    sim = FreewaySim(cfg)
    res = sim.run()
    m, tail, qlen = compute_metrics(res, cfg)
    m.update(tag)
    maps = dict(V=res["V"].astype(np.float32), rho=res["rho"].astype(np.float32), Q=res["Q"].astype(np.float32),
                qlen=qlen.astype(np.float32), tail=tail.astype(np.float32), det_bn=res["det_bn"], pend_ts=res["pend_ts"])
    return m, maps


def _parallel_map(fn, jobs, n_jobs):
    if HAVE_JOBLIB and n_jobs != 1 and len(jobs) > 1:
        try:
            gen = Parallel(n_jobs=n_jobs, return_as="generator")(delayed(fn)(j) for j in jobs)
            return list(tqdm(gen, total=len(jobs)))
        except TypeError:                       # old joblib without return_as
            return Parallel(n_jobs=n_jobs)(delayed(fn)(j) for j in tqdm(jobs))
    return [fn(j) for j in tqdm(jobs)]


def run_experiment(base: SimConfig, penetrations, distances, n_reps=5, n_jobs=-1, seed0=1000,
                   include_baseline=True, keep_maps=True):
    """Full-factorial experiment: penetration x disturbance distance x replicate (+ matching no-disturbance baselines).
    Common random numbers: replicate r uses the same arrival stream for every penetration and every distance, so
    a disturbed run and its baseline differ ONLY by the disturbance (and AV sets are nested across penetrations)."""
    jobs = []
    for rep in range(n_reps):
        for p in penetrations:
            dists = ([None] if include_baseline else []) + list(distances)
            for d in dists:
                cfg = replace(base, av_penetration=float(p), dist_distance=d, seed=seed0 + rep)
                tag = dict(pen=float(p), dist=np.nan if d is None else float(d), disturbed=d is not None, rep=rep)
                jobs.append((cfg, tag))
    t0 = time.time()
    out = _parallel_map(_run_one, jobs, n_jobs)
    print(f"{len(jobs)} runs finished in {(time.time() - t0) / 60:.1f} min")
    df = pd.DataFrame([o[0] for o in out])
    maps = {}
    if keep_maps:
        for (m, mp) in out:
            maps[(m["pen"], None if not m["disturbed"] else m["dist"], m["rep"])] = mp
    return df, maps


def estimate_capacity(base: SimConfig, penetrations, n_reps=2, n_jobs=-1, load_demand=2400.0, t_from=300.0, seed0=500):
    """Saturated-demand runs: the discharge flow past the lane drop while a queue exists = bottleneck capacity C(p)
    (vehicles/hour through the remaining lanes, including ramp traffic)."""
    jobs = []
    for rep in range(n_reps):
        for p in penetrations:
            cfg = replace(base, av_penetration=float(p), dist_distance=None, demand_per_lane=load_demand, seed=seed0 + rep,
                          t_dist=t_from)
            jobs.append((cfg, dict(pen=float(p), rep=rep)))
    out = _parallel_map(_run_one, jobs, n_jobs)
    df = pd.DataFrame([o[0] for o in out])
    cap = df.groupby("pen")["thru_vph"].agg(["mean", "std"]).rename(columns={"mean": "capacity_vph", "std": "sd"})
    cap["capacity_vph_lane"] = cap["capacity_vph"] / (base.n_lanes - 1)
    return cap


def demand_for_load(capacity_vph, load, ramp_demand, n_lanes=3):
    """Upstream demand per lane such that (3 lanes of mainline + ramp) = load * capacity."""
    return max((load * capacity_vph - ramp_demand) / n_lanes, 0.0)


def demand_sweep(base: SimConfig, penetrations, demands, n_reps=2, n_jobs=-1, seed0=700):
    """Baseline (no disturbance) runs over a demand grid -> where does recurrent congestion appear?"""
    jobs = []
    for rep in range(n_reps):
        for p in penetrations:
            for q in demands:
                cfg = replace(base, av_penetration=float(p), dist_distance=None, demand_per_lane=float(q), seed=seed0 + rep)
                jobs.append((cfg, dict(pen=float(p), demand=float(q), rep=rep)))
    out = _parallel_map(_run_one, jobs, n_jobs)
    return pd.DataFrame([o[0] for o in out])

## 6. Analysis and plotting helpers

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    ph = k / n
    den = 1 + z * z / n
    c = (ph + z * z / (2 * n)) / den
    h = z * math.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / den
    return (max(0.0, c - h), min(1.0, c + h))


def pair_with_baseline(df):
    """Attach the matching no-disturbance run (same penetration, same replicate/seed) to every disturbed run."""
    base = df[~df.disturbed][["pen", "rep", "q_max", "spill", "t_diss", "ttt_h", "thru_vph", "q_pre", "mean_tt"]]
    base = base.rename(columns={c: c + "_base" for c in base.columns if c not in ("pen", "rep")})
    d = df[df.disturbed].merge(base, on=["pen", "rep"], how="left")
    d["dq_max"] = d["q_max"] - d["q_max_base"]
    d["dttt_h"] = d["ttt_h"] - d["ttt_h_base"]
    d["dmean_tt"] = d["mean_tt"] - d["mean_tt_base"]
    d["spill_attr"] = d["spill"] & ~d["spill_base"].astype(bool)     # spillback that exists only because of the disturbance
    return d


def summarize(dd):
    """Cell means over replicates for every (penetration, distance)."""
    rows = []
    for (p, d), g in dd.groupby(["pen", "dist"]):
        k, n = int(g.spill.sum()), len(g)
        lo, hi = wilson(k, n)
        ka = int(g.spill_attr.sum())
        rows.append(dict(pen=p, dist=d, n=n, P_spill=k / n, P_lo=lo, P_hi=hi, P_attr=ka / n,
                         q_max=g.q_max.mean(), q_max_sd=g.q_max.std(), dq_max=g.dq_max.mean(),
                         t_diss=g.t_diss.mean(), diss_cens=g.diss_censored.mean(),
                         shock_kmh=g.shock_kmh.mean(), t_spill=g.t_spill.mean(),
                         dttt_h=g.dttt_h.mean(), thru_vph=g.thru_vph.mean(), q_base=g.q_max_base.mean()))
    return pd.DataFrame(rows).sort_values(["dist", "pen"]).reset_index(drop=True)


def required_penetration(S, eps=0.10, col="P_spill"):
    """For each distance: smallest penetration p* such that P(spillback) <= eps at p* and at every higher penetration.
    NaN = not reached within the simulated range (even the highest penetration is not enough)."""
    out = {}
    for d, g in S.groupby("dist"):
        g = g.sort_values("pen")
        ok = (g[col] <= eps).values
        pstar = np.nan
        for i in range(len(g)):
            if ok[i:].all():
                pstar = g.pen.values[i]
                break
        out[d] = pstar
    return pd.Series(out, name=f"p_star(eps={eps})").rename_axis("dist")


def critical_distance(S, eps=0.10, col="P_spill"):
    """Dual view: for each penetration, the smallest distance d* such that the disturbance causes spillback with
    probability <= eps at d* and at all larger distances (= 'safe buffer length'). NaN = no tested distance is safe."""
    out = {}
    for p, g in S.groupby("pen"):
        g = g.sort_values("dist")
        ok = (g[col] <= eps).values
        dstar = np.nan
        for i in range(len(g)):
            if ok[i:].all():
                dstar = g.dist.values[i]
                break
        out[p] = dstar
    return pd.Series(out, name=f"d_star(eps={eps})").rename_axis("pen")


def bootstrap_required(dd, eps=0.10, B=500, seed=0, col="spill"):
    """Bootstrap over replicates -> uncertainty of p*(d)."""
    rng = np.random.default_rng(seed)
    pens = sorted(dd.pen.unique()); dists = sorted(dd.dist.unique())
    cells = {(p, d): dd[(dd.pen == p) & (dd.dist == d)][col].astype(float).values for p in pens for d in dists}
    res = {d: [] for d in dists}
    for _ in range(B):
        for d in dists:
            probs = []
            for p in pens:
                a = cells[(p, d)]
                probs.append(rng.choice(a, size=len(a), replace=True).mean() if len(a) else np.nan)
            ok = np.array(probs) <= eps
            ps = np.nan
            for i in range(len(pens)):
                if ok[i:].all():
                    ps = pens[i]; break
            res[d].append(ps)
    rows = []
    for d in dists:
        a = np.array(res[d], float)
        rows.append(dict(dist=d, p_star_mean=np.nanmean(a) if np.isfinite(a).any() else np.nan,
                         p_star_lo=np.nanpercentile(a, 2.5) if np.isfinite(a).any() else np.nan,
                         p_star_hi=np.nanpercentile(a, 97.5) if np.isfinite(a).any() else np.nan,
                         share_not_reached=float(np.isnan(a).mean())))
    return pd.DataFrame(rows)


def _ols(X, y, names):
    X = np.asarray(X, float); y = np.asarray(y, float)
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    dof = max(len(y) - X.shape[1], 1)
    s2 = resid @ resid / dof
    cov = s2 * np.linalg.pinv(X.T @ X)
    se = np.sqrt(np.diag(cov))
    t = beta / np.where(se > 0, se, np.nan)
    p = 2 * (1 - stats.t.cdf(np.abs(t), dof))
    r2 = 1 - (resid @ resid) / ((y - y.mean()) @ (y - y.mean()) + 1e-12)
    return pd.DataFrame(dict(coef=beta, se=se, t=t, p=p), index=names), r2


def fit_models(dd, response="q_max"):
    """Does distance matter, does penetration matter, and do they interact?
    response ~ pen + dist_km + pen:dist_km   (OLS), and logit(spill) ~ same if statsmodels is installed."""
    d = dd.copy()
    d["dist_km"] = d.dist / 1000.0
    out = {}
    X = np.column_stack([np.ones(len(d)), d.pen, d.dist_km, d.pen * d.dist_km])
    tab, r2 = _ols(X, d[response], ["const", "pen", "dist_km", "pen:dist_km"])
    out["ols"] = (tab, r2)
    if HAVE_SM:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                m = smf.glm("spill ~ pen + dist_km + pen:dist_km", data=d.assign(spill=d.spill.astype(int)),
                            family=sm.families.Binomial()).fit()
            out["logit"] = m.summary2().tables[1]
        except Exception as e:
            out["logit"] = f"logit failed ({type(e).__name__}: {e}) - usually (quasi-)perfect separation; use more replicates or a finer penetration grid"
    rho = {}
    for p, g in d.groupby("pen"):
        if g.dist.nunique() > 2 and g[response].nunique() > 1:
            r, pv = stats.spearmanr(g.dist, g[response])
            rho[p] = (r, pv)
    out["spearman_by_pen"] = pd.DataFrame(rho, index=["rho", "p_value"]).T.rename_axis("pen")
    return out


def reclassify(df, maps, cfg: SimConfig, spill_len=None, v_queue=None, q_clear=None):
    """Re-derive q_max / spill from the STORED speed maps with a different queue definition (no re-simulation needed).
    Use it to test how sensitive the conclusions are to the spillback threshold or the congestion-speed threshold."""
    c2 = replace(cfg, **{k: v for k, v in dict(spill_len=spill_len, v_queue=v_queue, q_clear=q_clear).items() if v is not None})
    nt = next(iter(maps.values()))["V"].shape[0]
    tc = (np.arange(nt) + 0.5) * c2.dt_bin
    post = tc >= c2.t_dist
    rows = []
    for _, r in df.iterrows():
        key = (r.pen, None if not r.disturbed else r.dist, r.rep)
        _, q = queue_profile(maps[key]["V"].astype(float), c2)
        rows.append(float(q[post].max()))
    out = df.copy()
    out["q_max"] = rows
    out["spill"] = out["q_max"] >= c2.spill_len
    return out


def sweep_param(base: SimConfig, name, values, penetrations, distances, n_reps=3, n_jobs=-1, eps=0.10, seed0=2000):
    """Re-run the experiment for several values of one SimConfig field and report p*(d) for each value."""
    rows, tables = [], {}
    for val in values:
        cfg = replace(base, **{name: val})
        df, _ = run_experiment(cfg, penetrations, distances, n_reps=n_reps, n_jobs=n_jobs, seed0=seed0, keep_maps=False)
        dd = pair_with_baseline(df); S = summarize(dd)
        ps = required_penetration(S, eps)
        tables[val] = S
        for d, p in ps.items():
            rows.append({name: val, "dist": d, "p_star": p})
    return pd.DataFrame(rows), tables

In [ ]:
def plot_spacetime(mp, cfg, field="V", title="", ax=None, dist=None, vmax=None):
    """Space-time map. field: 'V' speed [km/h], 'rho' density [veh/km/lane], 'Q' flow [veh/h/lane]."""
    Z = mp[field].astype(float)
    if field == "V":
        Z = Z * 3.6; label = "speed [km/h]"; cmap = "RdYlGn"; vmax = vmax or cfg.human.v0 * 3.6
    elif field == "rho":
        label = "density [veh/km/lane]"; cmap = "magma_r"; vmax = vmax or 120
    else:
        label = "flow [veh/h/lane]"; cmap = "viridis"; vmax = vmax or 2500
    own = ax is None
    if own:
        fig, ax = plt.subplots(figsize=(8, 4.2))
    im = ax.imshow(Z.T, origin="lower", aspect="auto", cmap=cmap, vmin=0, vmax=vmax,
                   extent=[0, cfg.t_end, 0, cfg.L])
    ax.axhline(cfg.x_bn, color="k", lw=1.2, ls="--"); ax.text(cfg.t_end * 0.99, cfg.x_bn + 25, "lane drop", ha="right", fontsize=8)
    if cfg.ramp_enabled:
        ax.axhline(cfg.x_bn - cfg.ramp_offset, color="gray", lw=0.8, ls=":")
    if dist is not None and np.isfinite(dist):
        ax.plot([cfg.t_dist], [cfg.x_bn - dist], marker="v", color="blue", ms=9, label="disturbance")
        ax.legend(loc="upper left", fontsize=8)
    ax.axvline(cfg.t_dist, color="blue", lw=0.6, alpha=0.5)
    ax.set_xlabel("time [s]"); ax.set_ylabel("position x [m]"); ax.set_title(title, fontsize=10)
    ax.grid(False)
    plt.colorbar(im, ax=ax, label=label)
    if own:
        plt.tight_layout(); plt.show()
    return ax


def plot_map_grid(maps, cfg, pens, dist, rep=0, field="rho"):
    """One space-time panel per penetration for a given disturbance distance (the core picture of the study)."""
    fig, axs = plt.subplots(1, len(pens), figsize=(3.4 * len(pens), 3.8), sharey=True)
    axs = np.atleast_1d(axs)
    for ax, p in zip(axs, pens):
        plot_spacetime(maps[(float(p), dist, rep)], cfg, field, f"AV {int(round(p * 100))} %", ax=ax, dist=dist)
    fig.suptitle(f"{'density' if field == 'rho' else field} maps, disturbance {dist:.0f} m upstream of the lane drop (rep {rep})", y=1.02)
    plt.tight_layout(); plt.show()


def plot_queue_tails(maps, cfg, pens, dists, rep=0):
    """Queue length over time for every penetration (columns) and disturbance distance (lines)."""
    tc = (np.arange(int(math.ceil(cfg.t_end / cfg.dt_bin))) + 0.5) * cfg.dt_bin
    fig, axs = plt.subplots(1, len(pens), figsize=(3.3 * len(pens), 3.2), sharey=True)
    axs = np.atleast_1d(axs)
    cols = plt.cm.viridis(np.linspace(0, 0.9, len(dists)))
    for ax, p in zip(axs, pens):
        b = maps.get((float(p), None, rep))
        if b is not None:
            ax.plot(tc, b["qlen"], color="k", lw=1.4, ls="--", label="no disturbance")
        for c, d in zip(cols, dists):
            ax.plot(tc, maps[(float(p), float(d), rep)]["qlen"], color=c, lw=1.1, label=f"{d:.0f} m")
        ax.axhline(cfg.spill_len, color="r", lw=0.8, ls=":")
        ax.axvline(cfg.t_dist, color="b", lw=0.6, alpha=0.5)
        ax.set_title(f"AV {int(round(p * 100))} %", fontsize=10); ax.set_xlabel("time [s]")
    axs[0].set_ylabel("queue length [m]"); axs[-1].legend(fontsize=7, title="distance", loc="upper left")
    plt.tight_layout(); plt.show()


def plot_heatmap(S, col="P_spill", title=None, fmt="{:.2f}", cmap="RdYlGn_r", vmin=0, vmax=1, ax=None):
    piv = S.pivot(index="dist", columns="pen", values=col).sort_index(ascending=False)
    own = ax is None
    if own:
        fig, ax = plt.subplots(figsize=(6.4, 4.2))
    im = ax.imshow(piv.values, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels([f"{int(round(c * 100))}" for c in piv.columns])
    ax.set_yticks(range(piv.shape[0])); ax.set_yticklabels([f"{int(i)}" for i in piv.index])
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            val = piv.values[i, j]
            if np.isfinite(val):
                ax.text(j, i, fmt.format(val), ha="center", va="center", fontsize=8)
    ax.set_xlabel("AV penetration [%]"); ax.set_ylabel("disturbance distance upstream of lane drop [m]")
    ax.set_title(title or col); ax.grid(False)
    plt.colorbar(im, ax=ax)
    if own:
        plt.tight_layout(); plt.show()


def plot_required_penetration(pstar, boot=None, eps=0.1):
    fig, ax = plt.subplots(figsize=(6, 3.8))
    d = pstar.index.values.astype(float); p = pstar.values * 100
    ax.plot(d, np.where(np.isfinite(p), p, np.nan), "o-", color="C3", label="p* (point estimate)")
    nr = ~np.isfinite(p)
    if nr.any():
        ax.plot(d[nr], np.full(nr.sum(), 105), "x", color="C3", ms=9, label="not reached (>100 %)")
    if boot is not None:
        ax.fill_between(boot.dist, boot.p_star_lo * 100, boot.p_star_hi * 100, color="C3", alpha=0.15, label="95 % bootstrap")
    ax.set_xlabel("disturbance distance upstream of lane drop [m]"); ax.set_ylabel("required AV penetration p* [%]")
    ax.set_title(f"AV penetration needed so that P(spillback) <= {eps:.0%}"); ax.set_ylim(-5, 112); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()


def plot_metric_lines(S, metric, ylabel, ax=None, title=None):
    own = ax is None
    if own:
        fig, ax = plt.subplots(figsize=(6, 3.8))
    dists = sorted(S.dist.unique()); cols = plt.cm.viridis(np.linspace(0, 0.9, len(dists)))
    for c, d in zip(cols, dists):
        g = S[S.dist == d].sort_values("pen")
        ax.plot(g.pen * 100, g[metric], "o-", color=c, label=f"{d:.0f} m", ms=4)
    ax.set_xlabel("AV penetration [%]"); ax.set_ylabel(ylabel); ax.set_title(title or metric, fontsize=10)
    ax.legend(title="distance", fontsize=7)
    if own:
        plt.tight_layout(); plt.show()


def plot_fundamental_diagram(cfg, maps=None, pens=None):
    """Equilibrium IDM fundamental diagrams (human vs AV) and, optionally, simulated (rho, Q) cell values."""
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    for name, prm, c in (("human T=%.1f s" % cfg.human.T, cfg.human, "C0"), ("AV T=%.1f s" % cfg.av.T, cfg.av, "C3")):
        rho, Q, v = lane_capacity_idm(prm, cfg.veh_len)
        ax.plot(rho, Q, color=c, lw=2, label=f"IDM equilibrium, {name}  (cap {Q.max():.0f} veh/h/lane)")
    if maps is not None and pens is not None:
        cm = plt.cm.viridis(np.linspace(0, 0.9, len(pens)))
        nb = int(cfg.x_bn / cfg.dx) - 1
        for c, p in zip(cm, pens):
            key = [k for k in maps if k[0] == float(p) and k[1] is None]
            if not key:
                continue
            mp = maps[key[0]]
            r = mp["rho"][:, :nb].ravel(); q = mp["Q"][:, :nb].ravel()
            ok = np.isfinite(r) & (r > 0)
            ax.scatter(r[ok], q[ok], s=3, alpha=0.25, color=c, label=f"sim, AV {int(round(p * 100))} %")
    ax.set_xlabel("density [veh/km/lane]"); ax.set_ylabel("flow [veh/h/lane]"); ax.set_xlim(0, 160)
    ax.legend(fontsize=7); ax.set_title("Fundamental diagram: analytic IDM vs simulated cells")
    plt.tight_layout(); plt.show()

## 7. Study settings and validation
Pick `MODE = "quick"` while developing (a few minutes) and `"full"` for the numbers you report (6 penetrations × 5 distances × 10 replicates).

In [ ]:
# ---------------------------------------------------------------- run size ----------------------------------------------------------------
MODE = "quick"      # "quick" = fast exploratory pass | "full" = the design in the proposal
PRESETS = {
    "quick": dict(penetrations=[0.0, 0.4, 0.8, 1.0], distances=[250.0, 750.0, 1250.0], n_reps=3, t_end=900.0, t_dist=350.0),
    "full":  dict(penetrations=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
                  distances=[250.0, 500.0, 750.0, 1000.0, 1250.0], n_reps=10, t_end=1200.0, t_dist=400.0),
}
P = PRESETS[MODE]
PENETRATIONS, DISTANCES, N_REPS = P["penetrations"], P["distances"], P["n_reps"]
N_JOBS = max(1, (os.cpu_count() or 2) - 1)          # parallel worker processes; set to 1 if the hub restricts multiprocessing

# ---------------------------------------------------------------- base scenario (proposal Section 1 + Section 2) ----------------------------------------------------------------
BASE = SimConfig(
    L=3000.0, x_bn=1500.0, n_lanes=3,                # 3 km, 3 lanes -> 2 lanes at km 1.5
    human=IDMParams(T=1.2), av=IDMParams(T=0.6),     # v0=120 km/h, a=1.0, b=1.25, s0=1, s1=10, delta=4
    dist_factor=0.5,                                 # 50 % velocity-reduction pulse
    ramp_enabled=True, ramp_demand=300.0, ramp_storage=15,
    t_end=P["t_end"], t_dist=P["t_dist"],
)
print("MODE:", MODE, "|", len(PENETRATIONS), "penetrations x", len(DISTANCES), "distances x", N_REPS, "reps",
      "=", len(PENETRATIONS) * (len(DISTANCES) + 1) * N_REPS, "runs (incl. baselines)")
print("Total mainline demand in the proposal:", 3 * 1650, "veh/h  (+ ramp", BASE.ramp_demand, ")")

In [ ]:
# 1) Analytic IDM capacity of the two vehicle classes
for name, prm in (("human", BASE.human), ("AV", BASE.av)):
    rho, Q, v = lane_capacity_idm(prm, BASE.veh_len)
    print(f"{name:6s} T={prm.T:.1f}s : equilibrium capacity = {Q.max():6.0f} veh/h/lane at {rho[np.argmax(Q)]:5.1f} veh/km")
print("-> a lane drop (3->2) can carry at most 2x these numbers; real merging losses make it lower.\n")

# 2) Smoke run: one disturbed run, check physical consistency
cfg_t = replace(BASE, av_penetration=0.4, dist_distance=500.0, seed=1)
t0 = time.time()
sim = FreewaySim(cfg_t); res = sim.run(); m, tail, qlen = compute_metrics(res, cfg_t)
print(f"one run took {time.time() - t0:.1f} s")
print(f"minimum bumper gap seen: {res['min_gap']:.2f} m | overlap events repaired: {res['overlaps']} | lane changes: {res['n_lc']}")
assert res["overlaps"] == 0 and res["min_gap"] > 0, "collisions detected - check parameters / time step"
assert m["dist_applied"], "the disturbance was never applied"
print({k: (round(v, 2) if isinstance(v, float) else v) for k, v in m.items() if k in ("q_pre", "q_max", "t_diss", "shock_kmh", "thru_vph", "spill")})

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for ax, f in zip(axs, ("V", "rho", "Q")):
    plot_spacetime(res, cfg_t, f, {"V": "speed", "rho": "density", "Q": "flow"}[f] + " (AV 40 %, disturbance at 500 m)", ax=ax, dist=500.0)
plt.tight_layout(); plt.show()

**Check before continuing:** the three maps should show free flow upstream, a queue growing back from the lane drop (dashed line), no overlaps (`min gap > 0`). The equilibrium capacities printed above are the theoretical ceiling; the simulated discharge is lower because of merging.

## 8. Choose the demand regime — *important*
The proposal specifies 1,650 veh/h/lane × 3 lanes (+ ramp) ≈ 5,250 veh/h entering a 2-lane section. Even with 100 % AVs, two lanes carry roughly 4,200–4,600 veh/h in this model, so **the road is overloaded at every penetration**: a queue forms regardless of any disturbance and the answer to the focused question becomes trivial ("spillback always"). 

`estimate_capacity` measures the capacity C(p); `DEMAND_MODE="calibrated"` then sets demand to `LOAD_FACTOR × C(0 % AV)`, which is the regime where AVs can plausibly *prevent* spillback. Run the study with **both** modes and discuss the difference in the report.

In [ ]:
# Bottleneck capacity C(p) from saturated-demand runs (same horizon as the main study, discharge measured in the 2nd half)
cap = estimate_capacity(BASE, PENETRATIONS, n_reps=2, n_jobs=N_JOBS, t_from=BASE.t_end / 2)
display(cap.round(0))

proposal_total = 3 * 1650.0 + BASE.ramp_demand
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.bar(cap.index * 100, cap.capacity_vph, width=12, color="C0", label="bottleneck capacity C(p)")
ax.axhline(proposal_total, color="r", ls="--", label=f"proposal demand {proposal_total:.0f} veh/h")
ax.set_xlabel("AV penetration [%]"); ax.set_ylabel("veh/h through the 2 remaining lanes"); ax.legend(); plt.tight_layout(); plt.show()
if proposal_total > cap.capacity_vph.max():
    print("WARNING: the literal proposal demand exceeds capacity at EVERY penetration -> a persistent queue forms in all runs "
          "and the disturbance cannot be the cause. Use DEMAND_MODE = 'calibrated'.")

DEMAND_MODE = "calibrated"      # 'proposal' = 1650 veh/h/lane exactly as written | 'calibrated' = LOAD_FACTOR x C(0 % AV)
LOAD_FACTOR = 0.95              # try 0.85-1.05: lower = fewer baseline queues, higher = more recurrent congestion
if DEMAND_MODE == "proposal":
    lam = 1650.0
else:
    lam = demand_for_load(cap.loc[min(PENETRATIONS), "capacity_vph"], LOAD_FACTOR, BASE.ramp_demand, BASE.n_lanes)
BASE = replace(BASE, demand_per_lane=lam)
print(f"\nDEMAND_MODE={DEMAND_MODE}: using {BASE.demand_per_lane:.0f} veh/h/lane upstream (+{BASE.ramp_demand:.0f} veh/h ramp)")

In [ ]:
RUN_DEMAND_SWEEP = False        # optional (slow): baseline queue length vs demand for each penetration
if RUN_DEMAND_SWEEP:
    demands = np.arange(800, 1801, 200.0)
    ds = demand_sweep(BASE, PENETRATIONS, demands, n_reps=2, n_jobs=N_JOBS)
    tab = ds.groupby(["pen", "demand"]).q_max.mean().unstack(0)
    ax = tab.plot(marker="o", figsize=(6.5, 4)); ax.axhline(BASE.spill_len, color="r", ls=":")
    ax.set_xlabel("upstream demand [veh/h/lane]"); ax.set_ylabel("max queue length, no disturbance [m]"); ax.legend(title="AV share")
    plt.tight_layout(); plt.show()

## 9. Run the experiment
Every disturbed run is paired with a no-disturbance baseline that has *identical* arrivals, so differences are caused by the disturbance. Replicate *r* uses the same random stream for all penetrations (AV sets are nested).

In [ ]:
df, maps = run_experiment(BASE, PENETRATIONS, DISTANCES, n_reps=N_REPS, n_jobs=N_JOBS)

# quality control
dist_rows = df[df.disturbed]
print("disturbance applied in all disturbed runs:", bool(dist_rows.dist_applied.all()))
print("runs with vehicle overlaps (should be 0):", int((df.overlaps > 0).sum()), "| min gap overall: %.2f m" % df.min_gap.min())
print("runs where the virtual entry queue exceeded 100 vehicles:", int((df.pend_max > 100).sum()),
      "(demand cannot enter -> the queue reached the upstream boundary)")
os.makedirs("outputs", exist_ok=True)
df.to_csv("outputs/runs_raw.csv", index=False)
df.head()

## 10. Results: spillback probability, queue size, and the required penetration p*(d)

In [ ]:
EPS = 0.10          # tolerated spillback probability ("prevented" = P(spillback) <= EPS)

dd = pair_with_baseline(df)       # disturbed runs + their matching no-disturbance baseline (same random arrivals)
S = summarize(dd)
display(S.round(2))

fig, axs = plt.subplots(2, 2, figsize=(12, 8))
plot_heatmap(S, "P_spill", "P(spillback)  [queue >= %.0f m]" % BASE.spill_len, ax=axs[0, 0])
plot_heatmap(S, "P_attr", "P(spillback caused by the disturbance)\n(spillback in disturbed run but not in its baseline)", ax=axs[0, 1])
plot_heatmap(S, "q_max", "mean max queue length [m]", fmt="{:.0f}", cmap="viridis_r", vmin=0, vmax=BASE.x_bn, ax=axs[1, 0])
plot_heatmap(S, "dq_max", "disturbance effect on max queue [m]\n(disturbed - baseline)", fmt="{:+.0f}", cmap="coolwarm",
             vmin=-300, vmax=300, ax=axs[1, 1])
plt.tight_layout(); plt.show()

In [ ]:
pstar = required_penetration(S, EPS, "P_spill")            # the focused question of the proposal
pstar_attr = required_penetration(S, EPS, "P_attr")
dstar = critical_distance(S, EPS, "P_spill")               # dual: minimum safe buffer length per penetration
boot = bootstrap_required(dd, EPS, B=500)

print("Required AV penetration p* (smallest share keeping P(spillback) <= %.0f %%; NaN = not reached even at max penetration)" % (EPS * 100))
display(pd.concat([pstar.rename("p* (any spillback)"), pstar_attr.rename("p* (disturbance-caused spillback)")], axis=1))
print("Critical distance d*(p): shortest buffer for which spillback is prevented (NaN = none of the tested distances)")
display(dstar.to_frame())
plot_required_penetration(pstar, boot, EPS)

# --- hypothesis check: 'disturbances closer to the bottleneck require a HIGHER penetration'
ps = pstar.copy(); ps = ps.fillna(1.0 + 0.2)               # not reached -> treated as worse than 100 %
if ps.nunique() < 2:
    print("\np* is identical for every distance -> no distance effect detected at this demand level / sample size")
else:
    rho_s, p_s = stats.spearmanr(ps.index.values, ps.values)
    print(f"\nSpearman(distance, p*) = {rho_s:+.2f} (p = {p_s:.3f})  -> ", end="")
    print("SUPPORTS the prediction (larger distance, lower required penetration)" if rho_s < 0 and p_s < 0.1 else
          "does NOT clearly support the prediction at this sample size / demand level" if rho_s < 0 else "contradicts the prediction")

## 11. Queue dissipation, shockwave velocity, delay and statistical tests

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(16, 8))
plot_metric_lines(S, "q_max", "mean max queue length [m]", ax=axs[0, 0], title="Peak queue length")
plot_metric_lines(S, "dq_max", "disturbed - baseline [m]", ax=axs[0, 1], title="Queue growth caused by the disturbance")
plot_metric_lines(S, "t_diss", "s after disturbance", ax=axs[0, 2], title="Queue dissipation time (censored at run end)")
plot_metric_lines(S, "shock_kmh", "km/h (negative = upstream)", ax=axs[1, 0], title="Queue-tail shockwave velocity")
plot_metric_lines(S, "dttt_h", "veh-h", ax=axs[1, 1], title="Extra total travel time vs baseline")
plot_metric_lines(S, "thru_vph", "veh/h", ax=axs[1, 2], title="Discharge flow past the lane drop")
plt.tight_layout(); plt.show()

# --- statistical models: do distance, penetration and their interaction matter?
fm = fit_models(dd, "q_max")
print("OLS: q_max ~ pen + dist_km + pen:dist_km   (R^2 = %.2f)" % fm["ols"][1]); display(fm["ols"][0].round(3))
if "logit" in fm:
    print("Logit: P(spillback)"); display(fm["logit"] if not isinstance(fm["logit"], str) else print(fm["logit"]))
print("Spearman(distance, q_max) within each penetration:"); display(fm["spearman_by_pen"].round(3))

## 12. Space-time maps (ρ(x,t), speed), queue-tail trajectories, fundamental diagram

In [ ]:
REP = 0
for d in DISTANCES[:: max(1, len(DISTANCES) // 3)]:
    plot_map_grid(maps, BASE, PENETRATIONS, d, rep=REP, field="rho")      # rho(x,t) maps requested in the proposal
plot_map_grid(maps, BASE, PENETRATIONS, DISTANCES[0], rep=REP, field="V")
plot_queue_tails(maps, BASE, PENETRATIONS, DISTANCES, rep=REP)
plot_fundamental_diagram(BASE, maps, PENETRATIONS)

## 13. Sensitivity analyses
(a) is instant; (b) re-runs the model (set `RUN_SWEEPS = True`; use `MODE="quick"`-sized grids). Report whether p*(d) keeps its **ordering** under the alternatives — that is what makes the conclusion robust.

In [ ]:
# (a) Spillback definition: re-classify the stored runs with other thresholds (instant, no re-simulation)
for thr in (500.0, 750.0, 1000.0):
    d2 = pair_with_baseline(reclassify(df, maps, BASE, spill_len=thr))
    ps = required_penetration(summarize(d2), EPS)
    print(f"spill_len = {thr:5.0f} m ->", {int(k): v for k, v in ps.items()})
for vq in (30 / 3.6, 40 / 3.6, 60 / 3.6):
    d2 = pair_with_baseline(reclassify(df, maps, BASE, v_queue=vq))
    ps = required_penetration(summarize(d2), EPS)
    print(f"v_queue   = {vq * 3.6:4.0f} km/h ->", {int(k): v for k, v in ps.items()})

# (b) Model assumptions: re-run for other values of one parameter (SLOW - switch on deliberately)
RUN_SWEEPS = False
if RUN_SWEEPS:
    sw_pens, sw_dists = [0.0, 0.5, 1.0], [250.0, 1250.0]
    for name, vals in (("dist_duration", [5.0, 10.0, 20.0]),          # pulse length
                       ("dist_factor", [0.3, 0.5, 0.7]),               # pulse depth
                       ("ramp_enabled", [True, False]),                # ramp on/off
                       ("demand_per_lane", [BASE.demand_per_lane * f for f in (0.9, 1.0, 1.1)])):
        tab, _ = sweep_param(BASE, name, vals, sw_pens, sw_dists, n_reps=3, n_jobs=N_JOBS, eps=EPS)
        print("\n", name); display(tab.pivot(index="dist", columns=name, values="p_star"))

## 14. Export

In [ ]:
S.to_csv("outputs/summary_by_cell.csv", index=False)
dd.to_csv("outputs/runs_paired_with_baseline.csv", index=False)
pd.concat([pstar.rename("p_star_any"), pstar_attr.rename("p_star_attributable")], axis=1).to_csv("outputs/required_penetration.csv")
np.savez_compressed("outputs/density_maps.npz", **{f"pen{k[0]:.2f}_dist{'base' if k[1] is None else int(k[1])}_rep{k[2]}": v["rho"] for k, v in maps.items() if k[2] == 0})
print(sorted(os.listdir("outputs")))

## 15. Notes for the write-up
- **Interpreting p\*(d):** if p* is larger for short distances, the prediction holds (a short buffer leaves less room to absorb the pulse). If p* is flat, the AV benefit is capacity-driven (T = 0.6 s) rather than buffer-driven — also a valid finding.
- **Interpreting P_attr vs P_spill:** `P_spill` includes recurrent congestion (queues that would exist anyway); `P_attr` isolates the disturbance. Prefer `P_attr` for causal claims about distance.
- **Replicates:** with 10 replicates a probability has a resolution of 0.1 and wide Wilson intervals — state this, or raise `n_reps`.
- **Known limits:** no lane-specific AV behaviour, no communication delay, single disturbance per run, point-queue virtual entry buffer, fixed vehicle length.
- **Reproducibility:** seeds are `seed0 + replicate`; `outputs/runs_raw.csv` contains every run.